# Lipschitz constant of softmax v2

In [6]:
import torch
import torch.nn.functional as F

# Jacobian of softmax (differentiable)
def softmax_jacobian(s):
    s = s.view(-1, 1)
    return torch.diagflat(s) - s @ s.T 

seed = 1
ns = list(range(2, 16))   # vector sizes
steps = 1000
lr = 1e-1

for n in ns:
    g = torch.Generator().manual_seed(seed)
    x = (0.01 * torch.randn(n, generator=g, dtype=torch.float64)).requires_grad_(True)
    optimizer = torch.optim.Adam([x], lr=lr)

    for _ in range(steps):
        z = F.softmax(x, dim=0)              
        J = softmax_jacobian(z)              
        K = torch.linalg.norm(J, ord=2)      

        # gradient ascent on K
        optimizer.zero_grad()
        (-K).backward()                     
        optimizer.step()

    print(f"n={n:2d}  Lipschitz constant = {K.item():.6f} at x = {x.detach().numpy()}")

n= 2  Lipschitz constant = 0.500000 at x = [0.00464138 0.00464138]
n= 3  Lipschitz constant = 0.499942 at x = [ 2.3010564  -6.06074916  2.30106492]
n= 4  Lipschitz constant = 0.499934 at x = [ 4.46669444 -4.46703539 -4.46859431  4.46669954]
n= 5  Lipschitz constant = 0.499926 at x = [ 4.61106735 -4.60984938 -4.6114457   4.61107152 -4.6163448 ]
n= 6  Lipschitz constant = 0.499921 at x = [ 4.7256036  -4.72449608 -4.72613067  4.72560731 -4.7310491  -4.72826241]
n= 7  Lipschitz constant = 0.499919 at x = [ 4.82004171 -4.81778666 -4.81945614  4.82004474 -4.82439221 -4.8215998
 -4.83504202]
n= 8  Lipschitz constant = 0.499917 at x = [ 4.89857948 -4.90134294 -4.90305515  4.8985831  -4.90801267 -4.90521331
 -4.91866865 -4.90080782]
n= 9  Lipschitz constant = 0.499916 at x = [ 4.9690715  -4.9708182  -4.97256245  4.96907466 -4.97753613 -4.97473153
 -4.98819677 -4.97022132 -4.98325644]
n=10  Lipschitz constant = 0.499915 at x = [ 5.03139303 -5.03280945 -5.0345851   5.03139593 -5.03957457 -5.03676